# Edge AI: лёгкие нейросети на слабом железе

Практическая проверка того, о чём шла речь в реферате: лёгкая сеть из семейства MobileNet
и её поведение на устройствах разной мощности.

Шаги работы:
1. Загрузка MobileNetV2 и MobileNetV3-Small из torchvision.
2. Замер скорости на CPU и подсчёт числа параметров.
3. Экспорт в ONNX и Core ML, сравнение размеров.
4. Демонстрация INT8-квантизации.
5. Разбор того, почему MobileNetV3-Small не подходит для ESP32.

Стек: PyTorch + ONNX + Core ML, без TensorFlow. Все числа получены реальными запусками.

## 0. Подготовка

Установка зависимостей выполняется прямо из ноутбука — без команд в терминале. Ячейку ниже
достаточно выполнить один раз; если пакеты уже стоят, она просто ничего не меняет.

In [ ]:
!pip install -q "torch>=2.1" "torchvision>=0.16" "numpy>=1.24" "coremltools>=7.0" "onnx>=1.15" "onnxruntime>=1.16"

Note: you may need to restart the kernel to use updated packages.


In [ ]:
!pip install onnxscript

In [ ]:
import sys, os, time, platform, io, copy
import numpy as np
import torch
import torch.nn as nn
import torchvision

torch.manual_seed(0)
os.makedirs('artifacts', exist_ok=True)

/Users/aidar/miniconda3/envs/ds/lib/python3.11/site-packages/torchvision/io/image.py:14: UserWarning: Failed to load image Python extension: 'dlopen(/Users/aidar/miniconda3/envs/ds/lib/python3.11/site-packages/torchvision/image.so, 0x0006): Library not loaded: @rpath/libjpeg.9.dylib
  Referenced from: <EB3FF92A-5EB1-3EE8-AF8B-5923C1265422> /Users/aidar/miniconda3/envs/ds/lib/python3.11/site-packages/torchvision/image.so
  Reason: tried: '/Users/aidar/miniconda3/envs/ds/lib/python3.11/site-packages/torchvision/../../../libjpeg.9.dylib' (no such file), '/Users/aidar/miniconda3/envs/ds/lib/python3.11/site-packages/torchvision/../../../libjpeg.9.dylib' (no such file), '/Users/aidar/miniconda3/envs/ds/lib/python3.11/lib-dynload/../../libjpeg.9.dylib' (no such file), '/Users/aidar/miniconda3/envs/ds/bin/../lib/libjpeg.9.dylib' (no such file)'If you don't plan on using image functionality from `torchvision.io`, you can ignore this warning. Otherwise, there might be something wrong with your e

Python    : 3.11.11
PyTorch   : 2.5.1
torchvision: 0.20.1
Platform  : macOS-15.7.4-arm64-arm-64bit


## 1. Загрузка моделей

Две модели семейства MobileNet из torchvision, обе обучены на ImageNet.
Источник: https://github.com/pytorch/vision

In [4]:
from torchvision.models import (
    mobilenet_v2, MobileNet_V2_Weights,
    mobilenet_v3_small, MobileNet_V3_Small_Weights,
)

mnv2 = mobilenet_v2(weights=MobileNet_V2_Weights.IMAGENET1K_V1).eval()
mnv3 = mobilenet_v3_small(weights=MobileNet_V3_Small_Weights.IMAGENET1K_V1).eval()

def count_params(m):
    return sum(p.numel() for p in m.parameters())

print(f'MobileNetV2: {count_params(mnv2):>10,} параметров')
print(f'MobileNetV3-Small: {count_params(mnv3):>10,} параметров')

MobileNetV2:  3,504,872 параметров
MobileNetV3-Small:  2,542,856 параметров


## 2. Скорость на CPU

Модель прогоняется много раз на случайном входе; берётся среднее время одного прохода.
Это числа данного ноутбука.

In [5]:
def bench_cpu(model, n=50, warmup=10, size=224):
    x = torch.randn(1, 3, size, size)
    with torch.inference_mode():
        for _ in range(warmup):
            model(x)
        ts = []
        for _ in range(n):
            t0 = time.perf_counter()
            model(x)
            ts.append((time.perf_counter() - t0) * 1000)
    ts = np.array(ts)
    return float(ts.mean()), float(ts.std())

lat = {}
for name, m in [('MobileNetV2', mnv2), ('MobileNetV3-Small', mnv3)]:
    mean, std = bench_cpu(m)
    lat[name] = mean
    print(f'{name:18s} CPU: {mean:6.1f} +/- {std:4.1f} мс   ({1000/mean:5.1f} инф/с)')

MobileNetV2        CPU:   20.9 +/-  0.3 мс   ( 47.9 инф/с)
MobileNetV3-Small  CPU:   31.4 +/-  0.8 мс   ( 31.9 инф/с)


## 3. Экспорт в ONNX

ONNX — переносимый формат, он же затем подаётся в esp-ppq для ESP32. Заодно фиксируется
размер файла. Источник: https://pytorch.org/docs/stable/onnx.html

In [6]:
dummy = torch.randn(1, 3, 224, 224)

def export_onnx(model, path):
    torch.onnx.export(
        model, dummy, path,
        input_names=['input'], output_names=['logits'],
        opset_version=13,
    )
    return os.path.getsize(path) / 1e6

onnx_mb = {
    'MobileNetV2': export_onnx(mnv2, 'artifacts/mobilenet_v2.onnx'),
    'MobileNetV3-Small': export_onnx(mnv3, 'artifacts/mobilenet_v3_small.onnx'),
}
for k, v in onnx_mb.items():
    print(f'{k:18s} ONNX: {v:5.1f} МБ')

OnnxExporterError: Module onnx is not installed!

In [ ]:
# Если установлен onnxruntime — дополнительно замеряется скорость через него.
try:
    import onnxruntime as ort
    def bench_onnx(path, n=50, warmup=10):
        sess = ort.InferenceSession(path, providers=['CPUExecutionProvider'])
        x = np.random.randn(1, 3, 224, 224).astype(np.float32)
        iname = sess.get_inputs()[0].name
        for _ in range(warmup):
            sess.run(None, {iname: x})
        ts = []
        for _ in range(n):
            t0 = time.perf_counter()
            sess.run(None, {iname: x})
            ts.append((time.perf_counter() - t0) * 1000)
        return float(np.mean(ts)), float(np.std(ts))
    for name, p in [('MobileNetV2', 'artifacts/mobilenet_v2.onnx'),
                    ('MobileNetV3-Small', 'artifacts/mobilenet_v3_small.onnx')]:
        mean, std = bench_onnx(p)
        print(f'{name:18s} ONNX Runtime CPU: {mean:6.1f} +/- {std:4.1f} мс')
except ImportError:
    print('onnxruntime не установлен -> пропуск (pip install onnxruntime)')

## 4. Экспорт в Core ML

Тот же формат, что в приложениях на macOS и iPhone: на входе картинка, на выходе —
вероятности по классам ImageNet. Выполняется через coremltools, предпочтительно на macOS.
Источник: https://github.com/apple/coremltools

In [ ]:
try:
    import coremltools as ct

    MEAN = [0.485, 0.456, 0.406]; STD = [0.229, 0.224, 0.225]
    class Norm(nn.Module):
        def __init__(self, base):
            super().__init__(); self.base = base
            self.register_buffer('m', torch.tensor(MEAN).view(1, 3, 1, 1))
            self.register_buffer('s', torch.tensor(STD).view(1, 3, 1, 1))
        def forward(self, x):
            return torch.softmax(self.base((x - self.m) / self.s), dim=1)

    def dir_mb(path):
        return sum(os.path.getsize(os.path.join(r, f))
                   for r, _, fs in os.walk(path) for f in fs) / 1e6

    def export_coreml(model, weights, path):
        labels = weights.meta['categories']
        traced = torch.jit.trace(Norm(model).eval(), torch.rand(1, 3, 224, 224))
        img = ct.ImageType(name='image', shape=(1, 3, 224, 224),
                           scale=1/255.0, bias=[0, 0, 0], color_layout=ct.colorlayout.RGB)
        mlm = ct.convert(traced, inputs=[img],
                         classifier_config=ct.ClassifierConfig(class_labels=list(labels)),
                         convert_to='mlprogram')
        mlm.save(path)
        return dir_mb(path)

    coreml_mb = {
        'MobileNetV2': export_coreml(mnv2, MobileNet_V2_Weights.IMAGENET1K_V1, 'artifacts/MobileNetV2.mlpackage'),
        'MobileNetV3-Small': export_coreml(mnv3, MobileNet_V3_Small_Weights.IMAGENET1K_V1, 'artifacts/MobileNetV3Small.mlpackage'),
    }
    for k, v in coreml_mb.items():
        print(f'{k:18s} Core ML: {v:5.1f} МБ')
except ImportError:
    print('coremltools не установлен -> пропуск (pip install coremltools; лучше на macOS)')

## 5. Квантизация INT8 (PTQ)

INT8 — хранение весов в 8 битах вместо 32. Модель становится примерно вчетверо меньше и
быстрее на целочисленном железе; для микроконтроллера это решающе.

Ниже — быстрый пример динамической квантизации в PyTorch. Она затрагивает только слои
Linear, поэтому у MobileNet выигрыш по размеру небольшой. Полноценный INT8 для свёрток
выполняет esp-ppq при сборке модели для ESP32 (см. [../esp/quantize_mnv2.py](../esp/quantize_mnv2.py)).

In [ ]:
import torch.ao.quantization as tq

def state_dict_mb(m):
    b = io.BytesIO(); torch.save(m.state_dict(), b); return b.tell() / 1e6

print(f'MobileNetV2 fp32 state_dict : {state_dict_mb(mnv2):5.1f} МБ')
try:
    mnv2_int8 = tq.quantize_dynamic(copy.deepcopy(mnv2), {nn.Linear}, dtype=torch.qint8)
    print(f'MobileNetV2 int8 (dynamic)  : {state_dict_mb(mnv2_int8):5.1f} МБ')
except RuntimeError:
    # На Apple Silicon движок квантизации PyTorch обычно недоступен (NoQEngine).
    print('Динамическая квантизация недоступна на этом железе (нет QEngine).')
    print(f'Полный INT8 дал бы около {count_params(mnv2)/1e6:.1f} МБ (параметры x 1 байт) — это ~x4 меньше fp32.')
print('Настоящий INT8 для свёрток с калибровкой выполняет esp-ppq (см. ../esp/quantize_mnv2.py).')

## 6. Почему MobileNetV3-Small не подходит для ESP32

У MobileNetV3-Small есть блоки squeeze-and-excite: карта признаков домножается на
коэффициенты, посчитанные парой свёрток. После INT8-квантизации это умножение упирается в
аппаратное ограничение esp-dl на экспоненты:

```
output_exponent - input0_exponent - input1_exponent >= 0
```

и ломается на слое `/features/features.4/block/block.2/fc1/Conv`. Ни 16 бит, ни
выравнивание по слоям, ни weight-split не помогают без переобучения. Поэтому на ESP32
используется MobileNetV2 (там таких блоков нет), а MobileNetV3-Small остаётся для Mac и
iPhone. Ячейка ниже пересчитывает эти блоки в обеих моделях.

In [ ]:
from torchvision.ops import SqueezeExcitation

se_v3 = [m for m in mnv3.modules() if isinstance(m, SqueezeExcitation)]
se_v2 = [m for m in mnv2.modules() if isinstance(m, SqueezeExcitation)]
print(f'SE-блоков в MobileNetV3-Small: {len(se_v3)}')
print(f'SE-блоков в MobileNetV2      : {len(se_v2)}')

## 7. Итоги по устройствам

Скорость на CPU подставляется из ячеек выше. Остальные числа замерены прямо в приложениях
на каждом устройстве.

| Устройство | Модель | Точность | Латентность | FPS |
|---|---|---|---|---|
| Ноутбук (CPU, PyTorch) | MobileNetV2 | fp32 | см. §2 | — |
| Ноутбук (CPU, PyTorch) | MobileNetV3-Small | fp32 | см. §2 | — |
| MacBook Air M4 (Neural Engine) | MobileNetV3-Small | fp16 | 5–9 мс | ~30 |
| iPhone 15 (A16, Neural Engine) | MobileNetV3-Small | fp16 | 11–14 мс | ~30 |
| ESP32-S3 (ESP-DL) | MobileNetV2 | INT8 | ≈2754 мс | ~0,4 |

Главное видно сразу: на устройствах с нейронным ускорителем счёт идёт на миллисекунды, а на
микроконтроллере — почти три секунды на кадр. Разница между Mac и iPhone на глаз незаметна,
оба упираются в 30 кадров камеры.

In [ ]:
print('Скорость CPU на этом ноутбуке:')
for k, v in lat.items():
    print(f'  {k:18s}: {v:6.1f} мс')
print('\nНа устройствах замерено: Mac 5–9 мс, iPhone 11–14 мс, ESP32 ≈2754 мс (~0,4 FPS).')